In [3]:
# Imports
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Configuration
BASE_DIR = Path('/home/nucleon/herth-joh/scratch/SLIDERULE/MCNP/Pu/Screen/Mid')

print(f"📁 Répertoire de base : {BASE_DIR}")
print(f"📁 Existe : {BASE_DIR.exists()}")

📁 Répertoire de base : /home/nucleon/herth-joh/scratch/SLIDERULE/MCNP/Pu/Screen/Mid
📁 Existe : True


## 1. Fonctions d'extraction des données

In [4]:
def extract_metadata_from_folder_name(folder_name):
    """
    Extrait les métadonnées à partir du nom du dossier.
    
    Format attendu: SR_Pu_UN_G1_<config>_<particle>_D<dist>m_Lw_<lw>cm_Epw<epw>cm_w_<material>
    
    Returns:
        dict: Dictionnaire contenant les métadonnées extraites
    """
    pattern = r'SR_Pu_UN_G1_(C\d+)_(N|P)_D([\d.]+)m_Lw_([\d.]+)cm_Epw([\d.]+)cm_w_(\w+)'
    
    match = re.match(pattern, folder_name)
    if match:
        return {
            'config': match.group(1),
            'particle': match.group(2),
            'D_m': float(match.group(3)),
            'Lw_cm': float(match.group(4)),
            'Epw_cm': float(match.group(5)),
            'material': match.group(6)
        }
    return None


def parse_tally_from_output(output_file, tally_number):
    """
    Parse un tally depuis un fichier de sortie MCNP (.o).
    ATTENTION: Le fichier peut contenir plusieurs blocs pour le même tally
    (rapports intermédiaires à différents nps). Cette fonction extrait
    le DERNIER bloc (résultats finaux).
    
    Gère deux formats:
    1. Tally avec bins d'énergie (ex: tally 14 neutrons, tally 1014)
       - Format: energy   value   error puis ligne "total"
    2. Tally sans bins d'énergie (ex: tally 14 photons avec DE/DF)
       - Format: valeur directe sous "cell XXX" : value error
    
    Parameters:
        output_file: Chemin vers le fichier .o
        tally_number: Numéro du tally à extraire (14 ou 1014)
        
    Returns:
        dict: Dictionnaire contenant les données du tally:
              - 'total_value': valeur totale
              - 'total_error': erreur relative
              - 'energy_bins': liste des bornes d'énergie supérieures (MeV) ou []
              - 'values': liste des valeurs par bin ou []
              - 'errors': liste des erreurs relatives par bin ou []
    """
    with open(output_file, 'r') as f:
        content = f.read()
    
    # Pattern pour trouver le début du tally avec résultats
    # Format: "1tally       14        nps =    10000000"
    tally_pattern = rf'1tally\s+{tally_number}\s+nps\s*=\s*\d+'
    
    # Trouver TOUTES les occurrences et prendre la dernière
    all_matches = list(re.finditer(tally_pattern, content))
    if not all_matches:
        return None
    
    # Prendre le dernier match (résultats finaux)
    match = all_matches[-1]
    
    # Extraire la section du tally
    start_pos = match.start()
    
    # Trouver la fin de la section (prochain tally différent ou section statistique)
    # On cherche soit un autre numéro de tally, soit la section "===="
    next_section = re.search(r'\n1tally\s+(?!' + str(tally_number) + r'\s)|\n ====', content[start_pos + 100:])
    if next_section:
        end_pos = start_pos + 100 + next_section.start()
    else:
        end_pos = len(content)
    
    tally_section = content[start_pos:end_pos]
    
    # Extraire les données par bin d'énergie
    # Format: "    1.0000E-09   0.00000E+00 0.0000"
    # ou     "    1.0000E-09   1.23456E-07 0.1234"
    bin_pattern = r'^\s+([\d.E+-]+)\s+([\d.E+-]+)\s+([\d.]+)\s*$'
    
    # Pattern pour valeur directe (sans bins d'énergie, 2 colonnes seulement)
    # Format: "                                                   5.43753E-05 0.0008"
    direct_value_pattern = r'^\s+([\d.E+-]+)\s+([\d.]+)\s*$'
    
    energy_bins = []
    values = []
    errors = []
    total_value = None
    total_error = None
    
    in_cell_data = False
    has_energy_bins = 'energy' in tally_section.lower()
    
    for line in tally_section.split('\n'):
        # Détecter le début des données de cellule
        if 'cell' in line.lower() and 'volumes' not in line.lower():
            in_cell_data = True
            continue
        
        if in_cell_data:
            # Chercher la ligne "total"
            if 'total' in line.lower():
                total_match = re.search(r'total\s+([\d.E+-]+)\s+([\d.]+)', line, re.IGNORECASE)
                if total_match:
                    total_value = float(total_match.group(1))
                    total_error = float(total_match.group(2))
                break
            
            if has_energy_bins:
                # Chercher les données de bin (3 colonnes: energy, value, error)
                bin_match = re.match(bin_pattern, line)
                if bin_match:
                    energy_bins.append(float(bin_match.group(1)))
                    values.append(float(bin_match.group(2)))
                    errors.append(float(bin_match.group(3)))
            else:
                # Format sans bins d'énergie : valeur directe (2 colonnes: value, error)
                direct_match = re.match(direct_value_pattern, line)
                if direct_match and total_value is None:
                    total_value = float(direct_match.group(1))
                    total_error = float(direct_match.group(2))
                    break
    
    if total_value is None:
        return None
    
    return {
        'total_value': total_value,
        'total_error': total_error,
        'energy_bins': np.array(energy_bins),
        'values': np.array(values),
        'errors': np.array(errors)
    }


# Test sur un fichier neutron
test_file_n = BASE_DIR / 'C1_H_0/concrete/neutron/SR_Pu_UN_G1_C1_N_D1.0m_Lw_50.0cm_Epw1.0cm_w_concrete/SR_Pu_UN_G1_C1_N_D1.0m_Lw_50.0cm_Epw1.0cm_w_concrete.o'

print("Test extraction des tallies (dernier bloc = résultats finaux):")
print("="*60)

print("\n📌 Test NEUTRON:")
tally_14_n = parse_tally_from_output(test_file_n, 14)
if tally_14_n:
    print(f"  Tally 14: {tally_14_n['total_value']:.6e} ± {tally_14_n['total_error']:.4f} ({len(tally_14_n['energy_bins'])} bins)")
else:
    print("  Tally 14 non trouvé")

tally_1014_n = parse_tally_from_output(test_file_n, 1014)
if tally_1014_n:
    print(f"  Tally 1014: {tally_1014_n['total_value']:.6e} ± {tally_1014_n['total_error']:.4f} ({len(tally_1014_n['energy_bins'])} bins)")
else:
    print("  Tally 1014 non trouvé")

# Test sur un fichier photon (corrigé avec le bon nom C3)
test_file_p = BASE_DIR / 'C3_H_100/concrete/photon/SR_Pu_UN_G1_C3_P_D1.0m_Lw_50.0cm_Epw1.0cm_w_concrete/SR_Pu_UN_G1_C3_P_D1.0m_Lw_50.0cm_Epw1.0cm_w_concrete.o'

print("\n📌 Test PHOTON:")
tally_14_p = parse_tally_from_output(test_file_p, 14)
if tally_14_p:
    print(f"  Tally 14: {tally_14_p['total_value']:.6e} ± {tally_14_p['total_error']:.4f} ({len(tally_14_p['energy_bins'])} bins)")
else:
    print("  Tally 14 non trouvé")

tally_1014_p = parse_tally_from_output(test_file_p, 1014)
if tally_1014_p:
    print(f"  Tally 1014: {tally_1014_p['total_value']:.6e} ± {tally_1014_p['total_error']:.4f} ({len(tally_1014_p['energy_bins'])} bins)")
else:
    print("  Tally 1014 non trouvé")

Test extraction des tallies (dernier bloc = résultats finaux):

📌 Test NEUTRON:
  Tally 14: 1.729240e-04 ± 0.0062 (54 bins)
  Tally 1014: 6.222400e-06 ± 0.0176 (232 bins)

📌 Test PHOTON:
  Tally 14: 5.437530e-05 ± 0.0008 (0 bins)
  Tally 1014: 1.155540e-05 ± 0.0030 (67 bins)


## 2. Collecte des fichiers de sortie MCNP

In [5]:
def find_all_output_files(base_dir):
    """
    Trouve tous les fichiers de sortie MCNP (.o) dans l'arborescence.
    
    Returns:
        list: Liste de tuples (output_file_path, metadata_dict)
    """
    results = []
    
    base_path = Path(base_dir)
    
    # Parcourir les configurations (C1_H_0, C2_H_10, etc.)
    for config_dir in base_path.iterdir():
        if not config_dir.is_dir() or not config_dir.name.startswith('C'):
            continue
        
        # Extraire H depuis le nom du dossier (ex: C1_H_0 -> H=0)
        h_match = re.search(r'C\d+_H_(\d+)', config_dir.name)
        H_value = int(h_match.group(1)) if h_match else None
        
        # Parcourir les matériaux (concrete, lead, steel, water)
        for material_dir in config_dir.iterdir():
            if not material_dir.is_dir():
                continue
            
            material = material_dir.name
            if material not in ['concrete', 'lead', 'steel', 'water']:
                continue
            
            # Parcourir les types de particules (neutron, photon)
            for particle_dir in material_dir.iterdir():
                if not particle_dir.is_dir():
                    continue
                
                particle_type = particle_dir.name
                if particle_type not in ['neutron', 'photon']:
                    continue
                
                # Parcourir les simulations
                for sim_dir in particle_dir.iterdir():
                    if not sim_dir.is_dir():
                        continue
                    
                    # Extraire les métadonnées du nom du dossier
                    metadata = extract_metadata_from_folder_name(sim_dir.name)
                    if metadata is None:
                        continue
                    
                    # Ajouter H au metadata
                    metadata['H'] = H_value
                    metadata['config_dir'] = config_dir.name
                    
                    # Chercher le fichier .o
                    output_files = list(sim_dir.glob('*.o'))
                    if output_files:
                        results.append((output_files[0], metadata))
    
    return results


# Trouver tous les fichiers
print("🔍 Recherche des fichiers de sortie MCNP...")
output_files = find_all_output_files(BASE_DIR)
print(f"\n✓ Trouvé {len(output_files)} fichiers de sortie")

# Afficher un résumé
if output_files:
    print("\n📋 Exemple de fichier trouvé:")
    print(f"  Fichier: {output_files[0][0].name}")
    print(f"  Métadonnées: {output_files[0][1]}")

🔍 Recherche des fichiers de sortie MCNP...

✓ Trouvé 2600 fichiers de sortie

📋 Exemple de fichier trouvé:
  Fichier: SR_Pu_UN_G1_C2_P_D700.0m_Lw_35000.0cm_Epw20.0cm_w_water.o
  Métadonnées: {'config': 'C2', 'particle': 'P', 'D_m': 700.0, 'Lw_cm': 35000.0, 'Epw_cm': 20.0, 'material': 'water', 'H': 10, 'config_dir': 'C2_H_10'}

✓ Trouvé 2600 fichiers de sortie

📋 Exemple de fichier trouvé:
  Fichier: SR_Pu_UN_G1_C2_P_D700.0m_Lw_35000.0cm_Epw20.0cm_w_water.o
  Métadonnées: {'config': 'C2', 'particle': 'P', 'D_m': 700.0, 'Lw_cm': 35000.0, 'Epw_cm': 20.0, 'material': 'water', 'H': 10, 'config_dir': 'C2_H_10'}


## 3. Extraction des données de tous les fichiers

In [6]:
def process_all_outputs(output_files_list):
    """
    Traite tous les fichiers de sortie et crée un DataFrame.
    
    Returns:
        pd.DataFrame: DataFrame contenant toutes les données
    """
    results = []
    
    for output_file, metadata in tqdm(output_files_list, desc="Traitement des fichiers"):
        try:
            # Extraire les tallies
            tally_14 = parse_tally_from_output(output_file, 14)
            tally_1014 = parse_tally_from_output(output_file, 1014)
            
            # Créer l'entrée
            entry = {
                'file': output_file.name,
                'path': str(output_file),
                **metadata
            }
            
            # Tally 14 (Dp10)
            if tally_14:
                entry['tally_14_value'] = tally_14['total_value']
                entry['tally_14_error'] = tally_14['total_error']
            else:
                entry['tally_14_value'] = np.nan
                entry['tally_14_error'] = np.nan
            
            # Tally 1014 (universel - spectre)
            if tally_1014:
                entry['tally_1014_total_value'] = tally_1014['total_value']
                entry['tally_1014_total_error'] = tally_1014['total_error']
                entry['tally_1014_energy_bins'] = tally_1014['energy_bins']
                entry['tally_1014_values'] = tally_1014['values']
                entry['tally_1014_errors'] = tally_1014['errors']
            else:
                entry['tally_1014_total_value'] = np.nan
                entry['tally_1014_total_error'] = np.nan
                entry['tally_1014_energy_bins'] = None
                entry['tally_1014_values'] = None
                entry['tally_1014_errors'] = None
            
            results.append(entry)
            
        except Exception as e:
            print(f"\n⚠️  Erreur sur {output_file.name}: {e}")
    
    return pd.DataFrame(results)


# Traiter tous les fichiers
print("🔄 Extraction des données de tous les fichiers...")
print("="*60)

df_results = process_all_outputs(output_files)

print(f"\n✓ DataFrame créé avec {len(df_results)} lignes")
print(f"  Colonnes: {list(df_results.columns)}")

🔄 Extraction des données de tous les fichiers...


Traitement des fichiers: 100%|██████████| 2600/2600 [02:49<00:00, 15.38it/s]


✓ DataFrame créé avec 2600 lignes
  Colonnes: ['file', 'path', 'config', 'particle', 'D_m', 'Lw_cm', 'Epw_cm', 'material', 'H', 'config_dir', 'tally_14_value', 'tally_14_error', 'tally_1014_total_value', 'tally_1014_total_error', 'tally_1014_energy_bins', 'tally_1014_values', 'tally_1014_errors']


In [7]:
# Afficher un aperçu du DataFrame
print("📊 Aperçu du DataFrame:")
print("="*80)

# Colonnes à afficher (sans les spectres)
display_cols = ['file', 'config', 'H', 'particle', 'material', 'D_m', 'Epw_cm', 
                'tally_14_value', 'tally_14_error']
display_cols = [c for c in display_cols if c in df_results.columns]

df_results[display_cols].head(10)

📊 Aperçu du DataFrame:


,file,config,H,particle,material,D_m,Epw_cm,tally_14_value,tally_14_error
0,SR_Pu_UN_G1_C2_P_D700.0m_Lw_35000.0cm_Epw20.0c...,C2,10,P,water,700.0,20.0,2.349380e-12,0.2062
1,SR_Pu_UN_G1_C2_P_D700.0m_Lw_35000.0cm_Epw1.0cm...,C2,10,P,water,700.0,1.0,8.995020e-12,0.1476
2,SR_Pu_UN_G1_C2_P_D5.0m_Lw_250.0cm_Epw5.0cm_w_w...,C2,10,P,water,5.0,5.0,1.252470e-06,0.0033
3,SR_Pu_UN_G1_C2_P_D700.0m_Lw_35000.0cm_Epw10.0c...,C2,10,P,water,700.0,10.0,5.449080e-12,0.1879
4,SR_Pu_UN_G1_C2_P_D500.0m_Lw_25000.0cm_Epw40.0c...,C2,10,P,water,500.0,40.0,3.419960e-12,0.1428
5,SR_Pu_UN_G1_C2_P_D500.0m_Lw_25000.0cm_Epw20.0c...,C2,10,P,water,500.0,20.0,1.404240e-11,0.1951
6,SR_Pu_UN_G1_C2_P_D500.0m_Lw_25000.0cm_Epw1.0cm...,C2,10,P,water,500.0,1.0,4.259410e-11,0.2847
7,SR_Pu_UN_G1_C2_P_D300.0m_Lw_15000.0cm_Epw5.0cm...,C2,10,P,water,300.0,5.0,1.668340e-10,0.1644
8,SR_Pu_UN_G1_C2_P_D500.0m_Lw_25000.0cm_Epw10.0c...,C2,10,P,water,500.0,10.0,1.503990e-11,0.1179
9,SR_Pu_UN_G1_C2_P_D50.0m_Lw_2500.0cm_Epw40.0cm_...,C2,10,P,water,50.0,40.0,1.720100e-09,0.0261


In [8]:
# Vérifier les valeurs manquantes pour tally_14
print("Vérification des valeurs manquantes:")
print(f"  tally_14_value: {df_results['tally_14_value'].isna().sum()} / {len(df_results)}")
print(f"  tally_14_error: {df_results['tally_14_error'].isna().sum()} / {len(df_results)}")
print(f"  tally_1014_total_value: {df_results['tally_1014_total_value'].isna().sum()} / {len(df_results)}")
print()
print("Exemples de valeurs tally_14:")
print(df_results[['file', 'particle', 'tally_14_value', 'tally_14_error']].head(10))

Vérification des valeurs manquantes:
  tally_14_value: 0 / 2600
  tally_14_error: 0 / 2600
  tally_1014_total_value: 0 / 2600

Exemples de valeurs tally_14:
                                                file particle  tally_14_value  \
0  SR_Pu_UN_G1_C2_P_D700.0m_Lw_35000.0cm_Epw20.0c...        P    2.349380e-12   
1  SR_Pu_UN_G1_C2_P_D700.0m_Lw_35000.0cm_Epw1.0cm...        P    8.995020e-12   
2  SR_Pu_UN_G1_C2_P_D5.0m_Lw_250.0cm_Epw5.0cm_w_w...        P    1.252470e-06   
3  SR_Pu_UN_G1_C2_P_D700.0m_Lw_35000.0cm_Epw10.0c...        P    5.449080e-12   
4  SR_Pu_UN_G1_C2_P_D500.0m_Lw_25000.0cm_Epw40.0c...        P    3.419960e-12   
5  SR_Pu_UN_G1_C2_P_D500.0m_Lw_25000.0cm_Epw20.0c...        P    1.404240e-11   
6  SR_Pu_UN_G1_C2_P_D500.0m_Lw_25000.0cm_Epw1.0cm...        P    4.259410e-11   
7  SR_Pu_UN_G1_C2_P_D300.0m_Lw_15000.0cm_Epw5.0cm...        P    1.668340e-10   
8  SR_Pu_UN_G1_C2_P_D500.0m_Lw_25000.0cm_Epw10.0c...        P    1.503990e-11   
9  SR_Pu_UN_G1_C2_P_D50.0m_Lw_250

In [9]:
# Vérifier la répartition par type de particule
print("Valeurs manquantes par type de particule:")
for particle in df_results['particle'].unique():
    subset = df_results[df_results['particle'] == particle]
    missing = subset['tally_14_value'].isna().sum()
    print(f"  {particle}: {missing} / {len(subset)} manquantes")

# Vérifier un exemple de photon
print("\nExemple de fichier photon avec tally_14 manquant:")
photon_missing = df_results[(df_results['particle'] == 'P') & (df_results['tally_14_value'].isna())]
if len(photon_missing) > 0:
    example_file = photon_missing.iloc[0]['path']
    print(f"  Fichier: {example_file}")

Valeurs manquantes par type de particule:
  P: 0 / 1300 manquantes
  N: 0 / 1300 manquantes

Exemple de fichier photon avec tally_14 manquant:


In [10]:
# Statistiques par configuration
print("📈 Statistiques par configuration:")
print("="*80)

# Grouper par config et material
if 'config' in df_results.columns and 'material' in df_results.columns:
    stats = df_results.groupby(['config_dir', 'material', 'particle']).agg({
        'file': 'count',
        'tally_14_value': ['mean', 'std'],
        'tally_14_error': 'mean'
    }).round(6)
    stats.columns = ['N_simulations', 'Dose_mean', 'Dose_std', 'Error_mean']
    display(stats)

📈 Statistiques par configuration:


N_simulations  Dose_mean  Dose_std  Error_mean
config_dir material particle                                                
C1_H_0     concrete N                    65   0.000010  0.000031    0.067814
                    P                    65   0.000001  0.000002    0.066989
           lead     N                    65   0.000014  0.000038    0.033515
                    P                    65   0.000000  0.000001    0.028105
           steel    N                    65   0.000011  0.000032    0.026709
                    P                    65   0.000001  0.000002    0.037335
           water    N                    65   0.000008  0.000028    0.063443
                    P                    65   0.000001  0.000002    0.078786
C2_H_10    concrete N                    65   0.000005  0.000017    0.064935
                    P                    65   0.000002  0.000005    0.082005
           lead     N                    65   0.000007  0.000020    0.043017
                    P                    65   0.000001  0.000002    0.049615
           steel    N                    65   0.000006  0.000017    0.038560
                    P                    65   0.000001  0.000004    0.054000
           water    N                    65   0.000005  0.000015    0.087155
                    P                    65   0.000002  0.000005    0.088542
C3_H_100   concrete N                    65   0.000004  0.000014    0.064992
                    P                    65   0.000003  0.000010    0.090551
           lead     N                    65   0.000006  0.000016    0.042478
                    P                    65   0.000001  0.000004    0.048488
           steel    N                    64   0.000005  0.000014    0.038941
                    P                    65   0.000002  0.000007    0.063818
           water    N                    66   0.000004  0.000012    0.072776
                    P                    65   0.000004  0.000010    0.094769
C4_H_900   concrete N                    65   0.000003  0.000010    0.065786
                    P                    65   0.000003  0.000010    0.101755
           lead     N                    65   0.000004  0.000012    0.044760
                    P                    65   0.000001  0.000004    0.061269
           steel    N                    65   0.000003  0.000010    0.048362
                    P                    65   0.000002  0.000007    0.085015
           water    N                    65   0.000003  0.000009    0.086426
                    P                    65   0.000004  0.000011    0.093166
C5_H_2000  concrete N                    65   0.000002  0.000005    0.066722
                    P                    65   0.000002  0.000007    0.105474
           lead     N                    65   0.000002  0.000006    0.045649
                    P                    65   0.000001  0.000003    0.078066
           steel    N                    65   0.000002  0.000005    0.043763
                    P                    65   0.000001  0.000005    0.107952
           water    N                    65   0.000001  0.000004    0.103818
                    P                    65   0.000003  0.000008    0.099597

## 4. Coefficients de conversion dose (repris de postproc_mcnp.ipynb)

In [11]:
# ============================================================================
# COEFFICIENTS DE CONVERSION - NEUTRONS
# ============================================================================

# HPS N13.3 Dp(10) - Neutrons
# Source: ANSI/HPS N13.3-2013 Personal Dosimetry - Criteria for Testing
Dp10_HPS133_energy_bins_n = np.array([
    1e-9, 2.15e-9, 4.64e-9, 1e-8, 2.15e-8, 4.64e-8, 1e-7, 2.15e-7, 4.64e-7, 1e-6,
    2.15e-6, 4.64e-6, 1e-5, 2.15e-5, 4.64e-5, 1e-4, 2.15e-4, 4.64e-4, 1e-3, 2.15e-3,
    4.64e-3, 1e-2, 1.25e-2, 1.58e-2, 1.99e-2, 2.51e-2, 3.16e-2, 3.98e-2, 5.01e-2, 6.3e-2,
    7.94e-2, 0.1, 0.125, 0.158, 0.199, 0.251, 0.316, 0.398, 0.501, 0.63,
    0.794, 1.0, 1.25, 1.58, 1.99, 2.51, 3.16, 3.98, 5.01, 6.3,
    7.94, 10.0, 15.8, 20.0
])  # MeV

Dp10_HPS133_coeffs_n = np.array([
    1.00E-35, 2.00, 2.10, 2.30, 2.50, 2.60, 3.00, 3.40, 3.70, 3.70, 3.80,
    3.80, 3.90, 3.80, 3.70, 3.40, 3.40, 3.20, 3.20, 3.10, 3.10,
    3.20, 3.40, 3.50, 3.70, 3.90, 4.30, 4.70, 5.10, 5.80, 6.60,
    7.40, 8.30, 9.40, 10.90, 12.50, 14.20, 16.30, 19.70, 20.90, 23.10,
    26.80, 30.00, 32.60, 35.70, 37.40, 41.50, 50.80, 51.40, 54.30, 60.30,
    64.30, 72.90, 77.50
])  # pSv.cm²

# ICRP-74 (1996) - H*(10mm) [Neutrons]
# Source: TABLE A.42 p.200
Hstar10_icrp74_energy_bins_n = np.array([
    1.00E-09, 1.00E-08, 2.53E-08, 1.00E-07, 2.00E-07, 5.00E-07,
    1.00E-06, 2.00E-06, 5.00E-06, 1.00E-05, 2.00E-05, 5.00E-05,
    1.00E-04, 2.00E-04, 5.00E-04, 1.00E-03, 2.00E-03, 5.00E-03,
    1.00E-02, 2.00E-02, 3.00E-02, 5.00E-02, 7.00E-02, 1.00E-01,
    1.50E-01, 2.00E-01, 3.00E-01, 5.00E-01, 7.00E-01, 9.00E-01,
    1.00, 1.20, 2.00, 3.00, 4.00, 5.00,
    6.00, 7.00, 8.00, 9.00, 1.00E+01, 1.20E+01,
    1.40E+01, 1.50E+01, 1.60E+01, 1.80E+01, 2.00E+01, 3.00E+01,
    5.00E+01, 7.50E+01, 1.00E+02, 1.25E+02, 1.50E+02, 1.75E+02,
    2.01E+02
])  # MeV

Hstar10_icrp74_coeffs_n = np.array([
    6.60, 9.00, 10.6, 12.9, 13.5, 13.6,
    13.3, 12.9, 12.0, 11.3, 10.6, 9.90,
    9.40, 8.90, 8.30, 7.90, 7.70, 8.00,
    10.5, 16.6, 23.7, 41.1, 60.0, 88.0,
    132, 170, 233, 322, 375, 400,
    416, 425, 420, 412, 408, 405,
    400, 405, 409, 420, 440, 480,
    520, 540, 555, 570, 600, 515,
    400, 330, 285, 260, 245, 250,
    260
])  # pSv.cm²

# ICRP-116 (2010) - E [Neutrons]
# Source: TABLE A.5 p.130 (AP geometry)
E_ICRP116_energy_bins_n = np.array([
    1.00E-09, 1.00E-08, 2.50E-08, 1.00E-07, 2.00E-07,
    5.00E-07, 1.00E-06, 2.00E-06, 5.00E-06, 1.00E-05,
    2.00E-05, 5.00E-05, 1.00E-04, 2.00E-04, 5.00E-04,
    0.001, 0.002, 0.005, 0.01, 0.02,
    0.03, 0.05, 0.07, 0.1, 0.15,
    0.2, 0.3, 0.5, 0.7, 0.9,
    1, 1.2, 1.5, 2, 3,
    4, 5, 6, 7, 8,
    9, 10, 12, 14, 15,
    16, 18, 20, 21, 30,
    50, 75, 100, 130, 150,
    180, 200, 300, 400, 500,
    600, 700, 800, 900, 1000,
    2000, 5000, 10000
])  # MeV

E_ICRP116_coeffs_n = np.array([
    3.09, 3.55, 4, 5.2, 5.87,
    6.59, 7.03, 7.39, 7.71, 7.82,
    7.84, 7.82, 7.79, 7.73, 7.54,
    7.54, 7.61, 7.97, 9.11, 12.2,
    15.7, 23, 30.6, 41.9, 60.6,
    78.8, 114, 177, 232, 279,
    301, 330, 365, 407, 458,
    483, 494, 498, 499, 499,
    500, 500, 499, 495, 493,
    490, 484, 477, 474, 453,
    433, 420, 402, 382, 373,
    363, 359, 363, 389, 422,
    457, 486, 508, 524, 537,
    612, 716, 933
])  # pSv.cm²

# ============================================================================
# COEFFICIENTS DE CONVERSION - PHOTONS
# ============================================================================

# HPS N13.3 Dp(10) - Photons
# Source: ANSI/HPS N13.3-2013 Personal Dosimetry - Criteria for Testing
Dp10_HPS133_energy_bins_p = np.array([
    0.0100, 0.0125, 0.0150, 0.0175, 0.020,
    0.025, 0.030, 0.040, 0.050, 0.060,
    0.080, 0.10, 0.125, 0.15, 0.20,
    0.30, 0.40, 0.50, 0.60, 0.80,
    1.0, 1.5, 3.0, 6.0, 10.0
])  # MeV

Dp10_HPS133_coeffs_p = np.array([
    6.68700E-02, 4.49526E-01, 8.23680E-01, 1.13653E+00, 1.02648E+00,
    8.61808E-01, 8.01752E-01, 6.39210E-01, 5.70418E-01, 5.46788E-01,
    5.84221E-01, 6.71881E-01, 7.83552E-01, 9.62593E-01, 1.27715E+00,
    1.88922E+00, 2.45700E+00, 2.98928E+00, 3.48184E+00, 4.39110E+00,
    5.21649E+00, 6.99346E+00, 1.11253E+01, 1.78549E+01, 2.66640E+01
])  # pSv.cm²

# ICRP-74 (1996) - H*(10mm) [Photons]
# Source: TABLE A.21 p.179
Hstar10_icrp74_energy_bins_p = np.array([
    0.010, 0.015, 0.020, 0.030, 0.040, 0.050, 0.060,
    0.080, 0.100, 0.150, 0.200, 0.300, 0.400, 0.500,
    0.600, 0.800, 1, 1.5, 2, 3, 4,
    5, 6, 8, 10
])  # MeV

Hstar10_icrp74_coeffs_p = np.array([
    0.061, 0.83, 1.05, 0.81, 0.64, 0.55, 0.51,
    0.53, 0.61, 0.89, 1.20, 1.80, 2.38, 2.93,
    3.44, 4.38, 5.20, 6.90, 8.60, 11.1, 13.4,
    15.5, 17.6, 21.6, 25.6
])  # pSv.cm²

# ICRP-116 (2010) - E [Photons]
# Source: TABLE A.1 p.126 (AP geometry)
E_ICRP116_energy_bins_p = np.array([
    0.01, 0.015, 0.02, 0.03, 0.04,
    0.05, 0.06, 0.07, 0.08, 0.1,
    0.15, 0.2, 0.3, 0.4, 0.5,
    0.511, 0.6, 0.662, 0.8, 1,
    1.117, 1.33, 1.5, 2, 3,
    4, 5, 6, 6.129, 8,
    10, 15, 20, 30, 40,
    50, 60, 80, 100, 150,
    200, 300, 400, 500, 600,
    800, 1000, 1500, 2000, 3000,
    4000, 5000, 6000, 8000, 10000
])  # MeV

E_ICRP116_coeffs_p = np.array([
    0.0685, 0.156, 0.225, 0.313, 0.351,
    0.370, 0.390, 0.413, 0.444, 0.519,
    0.748, 1.00, 1.51, 2.00, 2.47,
    2.52, 2.91, 3.17, 3.73, 4.49,
    4.90, 5.59, 6.12, 7.48, 9.75,
    11.7, 13.4, 15.0, 15.1, 17.8,
    20.5, 26.1, 30.8, 37.9, 43.1,
    47.1, 50.1, 54.5, 57.8, 63.3,
    67.3, 72.3, 75.5, 77.5, 78.9,
    80.5, 81.7, 83.8, 85.2, 86.9,
    88.1, 88.9, 89.5, 90.2, 90.7
])  # pSv.cm²

# ============================================================================
# RÉSUMÉ DES COEFFICIENTS CHARGÉS
# ============================================================================
print("✓ Coefficients de conversion chargés:")
print("\n  NEUTRONS:")
print(f"    - HPS N13.3 Dp(10): {len(Dp10_HPS133_coeffs_n)} points")
print(f"    - ICRP-74 H*(10):   {len(Hstar10_icrp74_coeffs_n)} points")
print(f"    - ICRP-116 E:       {len(E_ICRP116_coeffs_n)} points")
print("\n  PHOTONS:")
print(f"    - HPS N13.3 Dp(10): {len(Dp10_HPS133_coeffs_p)} points")
print(f"    - ICRP-74 H*(10):   {len(Hstar10_icrp74_coeffs_p)} points")
print(f"    - ICRP-116 E:       {len(E_ICRP116_coeffs_p)} points")

# Vérifications de cohérence
assert len(Dp10_HPS133_energy_bins_n) == len(Dp10_HPS133_coeffs_n)
assert len(Hstar10_icrp74_energy_bins_n) == len(Hstar10_icrp74_coeffs_n)
assert len(E_ICRP116_energy_bins_n) == len(E_ICRP116_coeffs_n)
assert len(Dp10_HPS133_energy_bins_p) == len(Dp10_HPS133_coeffs_p)
assert len(Hstar10_icrp74_energy_bins_p) == len(Hstar10_icrp74_coeffs_p)
assert len(E_ICRP116_energy_bins_p) == len(E_ICRP116_coeffs_p)
print("\n  ✓ Vérification de cohérence OK")

✓ Coefficients de conversion chargés:

  NEUTRONS:
    - HPS N13.3 Dp(10): 54 points
    - ICRP-74 H*(10):   55 points
    - ICRP-116 E:       68 points

  PHOTONS:
    - HPS N13.3 Dp(10): 25 points
    - ICRP-74 H*(10):   25 points
    - ICRP-116 E:       55 points

  ✓ Vérification de cohérence OK


In [12]:
# ============================================================================
# FONCTION D'INTERPOLATION - Méthode trapézoïdale (par défaut)
# ============================================================================

from scipy.interpolate import interp1d

def interpolate_conversion_factors_trapezoidal(flux_energy_bins, coeff_energy_bins, coeff_values):
    """
    Interpole les coefficients de conversion pour les bins de flux.
    Méthode trapézoïdale : moyenne des valeurs aux bords du bin.
    
    Parameters:
    -----------
    flux_energy_bins : array
        Bornes supérieures des bins d'énergie du flux (MeV)
    coeff_energy_bins : array
        Énergies des coefficients de conversion (MeV)
    coeff_values : array
        Valeurs des coefficients de conversion
        
    Returns:
    --------
    interpolated_coeffs : array
        Coefficients interpolés pour chaque bin de flux
    """
    # Interpolation log-log
    log_interp = interp1d(
        np.log10(coeff_energy_bins),
        np.log10(coeff_values),
        kind='linear',
        bounds_error=False,
        fill_value='extrapolate'
    )
    
    # Construire les bornes inférieures
    E_low = np.concatenate([[0], flux_energy_bins[:-1]])
    E_high = flux_energy_bins
    
    # Pour chaque bin, calculer la moyenne des coefficients aux bords
    interpolated_coeffs = []
    for e_low, e_high in zip(E_low, E_high):
        if e_low <= 0:
            e_low = coeff_energy_bins[0] * 0.1  # Éviter log(0)
        
        c_low = 10**log_interp(np.log10(e_low))
        c_high = 10**log_interp(np.log10(e_high))
        c_avg = (c_low + c_high) / 2
        interpolated_coeffs.append(c_avg)
    
    return np.array(interpolated_coeffs)


def calculate_dose_with_uncertainty(flux_energy_bins, flux_values, flux_errors,
                                     coeff_energy_bins, coeff_values, 
                                     method='trapezoidal'):
    """
    Calcule la dose et son incertitude à partir du spectre de flux.
    
    D = Σ c_i × φ_i
    σ_D² = Σ (c_i × σ_φi)²
    
    Parameters:
    -----------
    flux_energy_bins : array
        Bornes supérieures des bins d'énergie (MeV)
    flux_values : array
        Valeurs du flux par bin
    flux_errors : array
        Erreurs relatives du flux par bin
    coeff_energy_bins : array
        Énergies des coefficients
    coeff_values : array
        Coefficients de conversion
    method : str
        Méthode d'interpolation ('trapezoidal' par défaut)
        
    Returns:
    --------
    dose : float
        Dose calculée
    dose_error : float
        Erreur absolue sur la dose
    interpolated_coeffs : array
        Coefficients interpolés utilisés
    """
    # Interpoler les coefficients
    interpolated_coeffs = interpolate_conversion_factors_trapezoidal(
        flux_energy_bins, coeff_energy_bins, coeff_values
    )
    
    # Calcul de la dose
    dose = np.sum(interpolated_coeffs * flux_values)
    
    # Propagation des erreurs (bins indépendants)
    flux_abs_errors = flux_values * flux_errors
    dose_variance = np.sum((interpolated_coeffs * flux_abs_errors)**2)
    dose_error = np.sqrt(dose_variance)
    
    return dose, dose_error, interpolated_coeffs


print("✓ Fonctions d'interpolation et de calcul de dose définies")

✓ Fonctions d'interpolation et de calcul de dose définies


## 5. Calcul des doses à partir du tally 1014

In [13]:
def compute_all_doses(df_results, method='trapezoidal'):
    """
    Calcule les 3 types de doses à partir du tally 1014 pour chaque simulation.
    
    Doses calculées :
    - Dp(10) HPS N13.3 : dose_Dp10_calc, dose_Dp10_calc_error
    - H*(10) ICRP-74   : dose_Hstar10_calc, dose_Hstar10_calc_error
    - E ICRP-116       : dose_E_calc, dose_E_calc_error
    
    Note: Les erreurs sont stockées en relatif (fraction), comme dans MCNP.
    """
    # Dictionnaires de coefficients par particule
    coeff_sets = {
        'N': {
            'Dp10': (Dp10_HPS133_energy_bins_n, Dp10_HPS133_coeffs_n),
            'Hstar10': (Hstar10_icrp74_energy_bins_n, Hstar10_icrp74_coeffs_n),
            'E': (E_ICRP116_energy_bins_n, E_ICRP116_coeffs_n)
        },
        'P': {
            'Dp10': (Dp10_HPS133_energy_bins_p, Dp10_HPS133_coeffs_p),
            'Hstar10': (Hstar10_icrp74_energy_bins_p, Hstar10_icrp74_coeffs_p),
            'E': (E_ICRP116_energy_bins_p, E_ICRP116_coeffs_p)
        }
    }
    
    # Initialiser les nouvelles colonnes
    dose_types = ['Dp10', 'Hstar10', 'E']
    for dose_type in dose_types:
        df_results[f'dose_{dose_type}_calc'] = np.nan
        df_results[f'dose_{dose_type}_calc_error'] = np.nan
    
    print(f"Calcul des doses pour {len(df_results)} simulations...")
    print("  Types de doses: Dp(10) HPS N13.3, H*(10) ICRP-74, E ICRP-116")
    print("="*80)
    
    n_success = 0
    n_failed = 0
    
    for idx, row in tqdm(df_results.iterrows(), total=len(df_results)):
        try:
            particle = row['particle']
            
            # Vérifier que les données tally 1014 existent
            energy_bins = row['tally_1014_energy_bins']
            if energy_bins is None or (isinstance(energy_bins, float) and np.isnan(energy_bins)):
                n_failed += 1
                continue
            
            # Vérifier qu'il y a des bins d'énergie (sinon pas de calcul possible)
            if len(energy_bins) == 0:
                n_failed += 1
                continue
            
            flux_values = row['tally_1014_values']
            flux_errors = row['tally_1014_errors']
            
            # Vérifier que la particule est connue
            if particle not in coeff_sets:
                n_failed += 1
                continue
            
            # Calculer les 3 types de doses
            for dose_type in dose_types:
                coeff_energy_bins, coeff_values = coeff_sets[particle][dose_type]
                
                dose, error, *_ = calculate_dose_with_uncertainty(
                    energy_bins, flux_values, flux_errors,
                    coeff_energy_bins, coeff_values,
                    method=method
                )
                
                # Enregistrer les résultats (erreurs en relatif)
                df_results.at[idx, f'dose_{dose_type}_calc'] = dose
                df_results.at[idx, f'dose_{dose_type}_calc_error'] = error / dose if dose != 0 else 0
            
            n_success += 1
                
        except Exception as e:
            n_failed += 1
            if n_failed <= 5:
                print(f"  ✗ Erreur sur {row['file']}: {e}")
    
    print("="*80)
    print(f"✓ Calculs terminés:")
    print(f"  Succès: {n_success}")
    print(f"  Échecs: {n_failed}")
    
    return df_results


# Calculer les doses
df_results = compute_all_doses(df_results)

Calcul des doses pour 2600 simulations...
  Types de doses: Dp(10) HPS N13.3, H*(10) ICRP-74, E ICRP-116


100%|██████████| 2600/2600 [01:04<00:00, 40.50it/s]

✓ Calculs terminés:
  Succès: 2600
  Échecs: 0


## 6. Validation : Comparaison dose MCNP vs dose calculée

In [14]:
# Statistiques des doses calculées

print("📊 STATISTIQUES DES DOSES CALCULÉES")
print("="*100)

# Liste des types de doses
dose_types = ['Dp10', 'Hstar10', 'E']
dose_labels = {
    'Dp10': 'Dp(10) HPS N13.3',
    'Hstar10': 'H*(10) ICRP-74',
    'E': 'E ICRP-116'
}

for particle, particle_name in [('N', 'NEUTRONS'), ('P', 'PHOTONS')]:
    df_particle = df_results[df_results['particle'] == particle]
    
    print(f"\n🔹 {particle_name} ({len(df_particle)} simulations)")
    print("-"*80)
    
    for dose_type in dose_types:
        col = f'dose_{dose_type}_calc'
        col_err = f'dose_{dose_type}_calc_error'
        
        df_valid = df_particle[df_particle[col].notna()]
        
        if len(df_valid) > 0:
            dose_mean = df_valid[col].mean()
            dose_min = df_valid[col].min()
            dose_max = df_valid[col].max()
            err_mean = df_valid[col_err].mean() * 100  # En %
            
            print(f"  {dose_labels[dose_type]:20s}: n={len(df_valid):4d}, "
                  f"mean={dose_mean:.2e}, range=[{dose_min:.2e}, {dose_max:.2e}], "
                  f"err_rel_moy={err_mean:.2f}%")
        else:
            print(f"  {dose_labels[dose_type]:20s}: Pas de données")

# Validation Dp10 : Comparaison avec MCNP (uniquement neutrons car DE/DF dans MCNP)
print("\n" + "="*100)
print("📊 VALIDATION Dp(10): Comparaison dose MCNP vs dose calculée")
print("="*100)

# Calculer les différences pour neutrons
df_neutrons = df_results[(df_results['particle'] == 'N') & 
                          df_results['dose_Dp10_calc'].notna() & 
                          df_results['tally_14_value'].notna()]

if len(df_neutrons) > 0:
    df_results.loc[df_neutrons.index, 'diff_Dp10_pct'] = (
        100 * (df_neutrons['dose_Dp10_calc'] - df_neutrons['tally_14_value']) / df_neutrons['tally_14_value']
    )
    
    diffs = df_results.loc[df_neutrons.index, 'diff_Dp10_pct']
    mean_diff = diffs.mean()
    std_diff = diffs.std()
    max_diff = diffs.abs().max()
    
    if max_diff < 0.5:
        verdict = "✅ EXCELLENT"
    elif max_diff < 2.0:
        verdict = "✅ BON"
    elif max_diff < 5.0:
        verdict = "⚠️  ACCEPTABLE"
    else:
        verdict = "❌ À VÉRIFIER"
    
    print(f"\n🔹 NEUTRONS ({len(df_neutrons)} simulations)")
    print(f"  Tally 14 vs calculé: μ={mean_diff:+6.2f}% σ={std_diff:5.2f}% max={max_diff:5.2f}% → {verdict}")
else:
    print("\nPas de données neutrons avec tally 14 pour validation")

print("\n💡 Note: Seule Dp(10) peut être comparée au tally 14 MCNP (même coefficients DE/DF)")
print("   H*(10) et E sont calculées indépendamment avec d'autres coefficients.")
print("="*100)

📊 STATISTIQUES DES DOSES CALCULÉES

🔹 NEUTRONS (1300 simulations)
--------------------------------------------------------------------------------
  Dp(10) HPS N13.3    : n=1300, mean=4.98e-06, range=[1.70e-15, 1.69e-04], err_rel_moy=5.62%
  H*(10) ICRP-74      : n=1300, mean=6.07e-05, range=[1.82e-14, 2.07e-03], err_rel_moy=5.57%
  E ICRP-116          : n=1300, mean=4.93e-05, range=[1.43e-14, 1.77e-03], err_rel_moy=6.66%

🔹 PHOTONS (1300 simulations)
--------------------------------------------------------------------------------
  Dp(10) HPS N13.3    : n=1300, mean=1.67e-06, range=[4.24e-15, 5.70e-05], err_rel_moy=7.56%
  H*(10) ICRP-74      : n=1300, mean=1.65e-06, range=[4.16e-15, 5.63e-05], err_rel_moy=7.60%
  E ICRP-116          : n=1300, mean=1.42e-06, range=[3.55e-15, 4.85e-05], err_rel_moy=7.67%

📊 VALIDATION Dp(10): Comparaison dose MCNP vs dose calculée

🔹 NEUTRONS (1300 simulations)
  Tally 14 vs calculé: μ= -4.38% σ= 0.46% max= 5.89% → ❌ À VÉRIFIER

💡 Note: Seule Dp(10) pe

In [15]:
# Afficher quelques exemples de résultats

print("\n📋 EXEMPLES DE RÉSULTATS DÉTAILLÉS")
print("="*100)

# Colonnes à afficher
display_cols_base = ['file', 'config', 'D_m', 'material', 'particle']
display_cols_doses = ['dose_Dp10_calc', 'dose_Hstar10_calc', 'dose_E_calc']
display_cols = [c for c in display_cols_base + display_cols_doses if c in df_results.columns]

# Quelques exemples neutrons
print("\n🔹 NEUTRONS (5 premiers exemples):")
df_examples_n = df_results[df_results['particle'] == 'N'][display_cols].head(5)
display(df_examples_n)

# Quelques exemples photons
print("\n🔹 PHOTONS (5 premiers exemples):")
df_examples_p = df_results[df_results['particle'] == 'P'][display_cols].head(5)
display(df_examples_p)

# Comparaison des 3 types de doses pour les neutrons
print("\n📊 COMPARAISON DES TYPES DE DOSES (Neutrons - 10 premiers):")
df_compare = df_results[df_results['particle'] == 'N'][
    ['file', 'config', 'dose_Dp10_calc', 'dose_Hstar10_calc', 'dose_E_calc']
].head(10)
display(df_compare)


📋 EXEMPLES DE RÉSULTATS DÉTAILLÉS

🔹 NEUTRONS (5 premiers exemples):


,file,config,D_m,material,particle,dose_Dp10_calc,dose_Hstar10_calc,dose_E_calc
65,SR_Pu_UN_G1_C2_N_D700.0m_Lw_35000.0cm_Epw20.0c...,C2,700.0,water,N,1.959030e-12,2.103646e-11,1.799729e-11
66,SR_Pu_UN_G1_C2_N_D700.0m_Lw_35000.0cm_Epw1.0cm...,C2,700.0,water,N,3.134695e-11,3.733749e-10,2.776555e-10
67,SR_Pu_UN_G1_C2_N_D5.0m_Lw_250.0cm_Epw5.0cm_w_w...,C2,5.0,water,N,3.502628e-06,3.869467e-05,3.321054e-05
68,SR_Pu_UN_G1_C2_N_D700.0m_Lw_35000.0cm_Epw10.0c...,C2,700.0,water,N,6.443686e-12,7.459136e-11,5.862052e-11
69,SR_Pu_UN_G1_C2_N_D500.0m_Lw_25000.0cm_Epw40.0c...,C2,500.0,water,N,8.232229e-13,8.911290e-12,7.743036e-12



🔹 PHOTONS (5 premiers exemples):


,file,config,D_m,material,particle,dose_Dp10_calc,dose_Hstar10_calc,dose_E_calc
0,SR_Pu_UN_G1_C2_P_D700.0m_Lw_35000.0cm_Epw20.0c...,C2,700.0,water,P,2.360938e-12,2.324308e-12,1.989403e-12
1,SR_Pu_UN_G1_C2_P_D700.0m_Lw_35000.0cm_Epw1.0cm...,C2,700.0,water,P,9.048810e-12,8.918610e-12,7.641652e-12
2,SR_Pu_UN_G1_C2_P_D5.0m_Lw_250.0cm_Epw5.0cm_w_w...,C2,5.0,water,P,1.254585e-06,1.241129e-06,1.071880e-06
3,SR_Pu_UN_G1_C2_P_D700.0m_Lw_35000.0cm_Epw10.0c...,C2,700.0,water,P,5.467667e-12,5.384471e-12,4.613551e-12
4,SR_Pu_UN_G1_C2_P_D500.0m_Lw_25000.0cm_Epw40.0c...,C2,500.0,water,P,3.439324e-12,3.391125e-12,2.885474e-12



📊 COMPARAISON DES TYPES DE DOSES (Neutrons - 10 premiers):


,file,config,dose_Dp10_calc,dose_Hstar10_calc,dose_E_calc
65,SR_Pu_UN_G1_C2_N_D700.0m_Lw_35000.0cm_Epw20.0c...,C2,1.959030e-12,2.103646e-11,1.799729e-11
66,SR_Pu_UN_G1_C2_N_D700.0m_Lw_35000.0cm_Epw1.0cm...,C2,3.134695e-11,3.733749e-10,2.776555e-10
67,SR_Pu_UN_G1_C2_N_D5.0m_Lw_250.0cm_Epw5.0cm_w_w...,C2,3.502628e-06,3.869467e-05,3.321054e-05
68,SR_Pu_UN_G1_C2_N_D700.0m_Lw_35000.0cm_Epw10.0c...,C2,6.443686e-12,7.459136e-11,5.862052e-11
69,SR_Pu_UN_G1_C2_N_D500.0m_Lw_25000.0cm_Epw40.0c...,C2,8.232229e-13,8.911290e-12,7.743036e-12
70,SR_Pu_UN_G1_C2_N_D500.0m_Lw_25000.0cm_Epw20.0c...,C2,1.083600e-11,1.109467e-10,9.904318e-11
71,SR_Pu_UN_G1_C2_N_D500.0m_Lw_25000.0cm_Epw1.0cm...,C2,1.585416e-10,1.750130e-09,1.376230e-09
72,SR_Pu_UN_G1_C2_N_D300.0m_Lw_15000.0cm_Epw5.0cm...,C2,3.623907e-10,4.209029e-09,3.175482e-09
73,SR_Pu_UN_G1_C2_N_D500.0m_Lw_25000.0cm_Epw10.0c...,C2,4.261223e-11,4.271959e-10,3.862528e-10
74,SR_Pu_UN_G1_C2_N_D50.0m_Lw_2500.0cm_Epw40.0cm_...,C2,5.312044e-10,5.369037e-09,4.946848e-09


## 7. Sauvegarde finale

In [16]:
# Sauvegarder les résultats complets (avec doses calculées)

# Supprimer les colonnes redondantes (anciennes colonnes remplacées par dose_Dp10_*)
cols_to_drop = ['dose_14_calc', 'dose_14_calc_error', 'diff_14_pct']
for col in cols_to_drop:
    if col in df_results.columns:
        df_results = df_results.drop(columns=[col])
        print(f"  ✗ Colonne redondante supprimée: {col}")

# Pickle (conserve les arrays numpy pour les spectres)
output_pkl = BASE_DIR / 'results_mcnp_Pu_Mid.pkl'
df_results.to_pickle(output_pkl)
print(f"✓ Résultats complets sauvegardés: {output_pkl}")

# CSV (sans les spectres, pour lecture facile)
cols_csv = [c for c in df_results.columns if 'energy_bins' not in c and 'values' not in c and 'errors' not in c]
cols_csv = [c for c in cols_csv if c != 'path']  # Exclure le chemin complet
output_csv = BASE_DIR / 'results_mcnp_Pu_Mid.csv'
df_results[cols_csv].to_csv(output_csv, index=False)
print(f"✓ Résultats CSV sauvegardés: {output_csv}")

print(f"\n📊 Résumé:")
print(f"  Total simulations: {len(df_results)}")
print(f"  Colonnes CSV: {len(cols_csv)}")
print(f"  Colonnes: {cols_csv}")

✓ Résultats complets sauvegardés: /home/nucleon/herth-joh/scratch/SLIDERULE/MCNP/Pu/Screen/Mid/results_mcnp_Pu_Mid.pkl
✓ Résultats CSV sauvegardés: /home/nucleon/herth-joh/scratch/SLIDERULE/MCNP/Pu/Screen/Mid/results_mcnp_Pu_Mid.csv

📊 Résumé:
  Total simulations: 2600
  Colonnes CSV: 20
  Colonnes: ['file', 'config', 'particle', 'D_m', 'Lw_cm', 'Epw_cm', 'material', 'H', 'config_dir', 'tally_14_value', 'tally_14_error', 'tally_1014_total_value', 'tally_1014_total_error', 'dose_Dp10_calc', 'dose_Dp10_calc_error', 'dose_Hstar10_calc', 'dose_Hstar10_calc_error', 'dose_E_calc', 'dose_E_calc_error', 'diff_Dp10_pct']
✓ Résultats CSV sauvegardés: /home/nucleon/herth-joh/scratch/SLIDERULE/MCNP/Pu/Screen/Mid/results_mcnp_Pu_Mid.csv

📊 Résumé:
  Total simulations: 2600
  Colonnes CSV: 20
  Colonnes: ['file', 'config', 'particle', 'D_m', 'Lw_cm', 'Epw_cm', 'material', 'H', 'config_dir', 'tally_14_value', 'tally_14_error', 'tally_1014_total_value', 'tally_1014_total_error', 'dose_Dp10_calc', 'dos